# 📓 Notebook 4: Adversarial Saldırılar

Bu notebook'ta:
- 3 farklı adversarial saldırı uygulayacağız
- Her model için performans düşüşünü ölçeceğiz
- Feature sensitivity analizi yapacağız
- Hangi model en dayanıklı sorusunu cevaplayacağız


In [1]:
import sys
sys.path.append('..')

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.adversarial import AdversarialAttacker, AdversarialEvaluator, AttackRunner
from src.utils import load_model, print_section, plot_metrics_comparison, FIGURES_DIR

print('✅ Import başarılı')

✅ Import başarılı


In [2]:
# Verileri yükle
X_test  = np.load('../data/processed/X_test.npy')
y_test  = np.load('../data/processed/y_test.npy')
X_train = np.load('../data/processed/X_train.npy')
y_train = np.load('../data/processed/y_train.npy')

with open('../data/processed/feature_names.json') as f:
    feature_names = json.load(f)

# Baseline modelleri yükle
baseline_models = {
    'Random Forest':       load_model('baseline_random_forest'),
    'XGBoost':             load_model('baseline_xgboost'),
    'Logistic Regression': load_model('baseline_logistic_regression'),
}

print(f'Test seti: {X_test.shape}')
print(f'Phishing: {y_test.sum()} | Legitimate: {(y_test==0).sum()}')
print(f'Modeller yüklendi: {list(baseline_models.keys())}')

[2026-02-28 12:38:07] INFO [utils] Model yüklendi: C:\Users\emira\OneDrive\Desktop\main projem\notebooks\..\data\models\baseline_random_forest.pkl
[2026-02-28 12:38:07] INFO [utils] Model yüklendi: C:\Users\emira\OneDrive\Desktop\main projem\notebooks\..\data\models\baseline_xgboost.pkl
[2026-02-28 12:38:07] INFO [utils] Model yüklendi: C:\Users\emira\OneDrive\Desktop\main projem\notebooks\..\data\models\baseline_logistic_regression.pkl


Test seti: (25013, 64)
Phishing: 12565 | Legitimate: 12448
Modeller yüklendi: ['Random Forest', 'XGBoost', 'Logistic Regression']


## 🔴 Saldırı 1: Minimal Feature Manipülasyonu

In [3]:
print_section('Saldırı 1: Minimal Feature Manipülasyonu')
print('Fikir: Phishing URL feature\'larını küçük değişimlerle meşru görünümlü hale getir')
print('Gerçek hayat: Saldırgan URL\'yi biraz modifiye eder (tire sil, uzunluğu azalt vs)')

attacker = AdversarialAttacker(manipulation_budget=5)
evaluator = AdversarialEvaluator()

X_adv_minimal = attacker.minimal_feature_manipulation(X_test, feature_names, y_test)

print(f'\nDeğiştirilen değer sayısı: {(X_test != X_adv_minimal).sum():,}')
print(f'Değiştirilen örnek sayısı: {(X_test != X_adv_minimal).any(axis=1).sum():,}')


════════════════════════════════════════════════════════════
  Saldırı 1: Minimal Feature Manipülasyonu
════════════════════════════════════════════════════════════

Fikir: Phishing URL feature'larını küçük değişimlerle meşru görünümlü hale getir
Gerçek hayat: Saldırgan URL'yi biraz modifiye eder (tire sil, uzunluğu azalt vs)

Değiştirilen değer sayısı: 37,113
Değiştirilen örnek sayısı: 12,476


In [4]:
# Her model için minimal saldırı değerlendir
print('\n📊 Minimal Manipulation Sonuçları:')
results_minimal = []
for name, model in baseline_models.items():
    r = evaluator.evaluate(model, X_test, X_adv_minimal, y_test, name)
    results_minimal.append(r)

df_minimal = pd.DataFrame(results_minimal).set_index('model')
print('\n', df_minimal[['acc_normal', 'acc_adversarial', 'acc_drop', 'attack_success_rate']].to_string())


📊 Minimal Manipulation Sonuçları:

  Adversarial Değerlendirme: Random Forest
  Normal Accuracy    : 0.9271
  Adversarial Accuracy: 0.9037
  Accuracy Düşüşü    : 0.0234 (2.5%)
  F1 Düşüşü          : 0.0259
  Saldırı Başarısı   : 5.09%
  Robustness Skoru   : 0.9766

  Adversarial Değerlendirme: XGBoost
  Normal Accuracy    : 0.9392
  Adversarial Accuracy: 0.9235
  Accuracy Düşüşü    : 0.0157 (1.7%)
  F1 Düşüşü          : 0.0169
  Saldırı Başarısı   : 3.34%
  Robustness Skoru   : 0.9843

  Adversarial Değerlendirme: Logistic Regression
  Normal Accuracy    : 0.8900
  Adversarial Accuracy: 0.7883
  Accuracy Düşüşü    : 0.1017 (11.4%)
  F1 Düşüşü          : 0.1310
  Saldırı Başarısı   : 23.76%
  Robustness Skoru   : 0.8983

                      acc_normal  acc_adversarial  acc_drop  attack_success_rate
model                                                                          
Random Forest            0.9271           0.9037    0.0234               0.0509
XGBoost                  0.9

## 🟠 Saldırı 2: Gaussian Gürültü

In [5]:
print_section('Saldırı 2: Gaussian Gürültü Saldırısı')
print('Fikir: Feature değerlerine küçük gürültü ekle')
print('Gerçek hayat: Saldırgan modelin feature\'larını tam bilmiyor, rastgele deniyor')

X_adv_gaussian = attacker.gaussian_noise_attack(
    X_test, feature_names, y_test, std_multiplier=2.0
)

results_gaussian = []
for name, model in baseline_models.items():
    r = evaluator.evaluate(model, X_test, X_adv_gaussian, y_test, name)
    results_gaussian.append(r)

df_gaussian = pd.DataFrame(results_gaussian).set_index('model')
print('\n', df_gaussian[['acc_normal', 'acc_adversarial', 'acc_drop', 'attack_success_rate']].to_string())


════════════════════════════════════════════════════════════
  Saldırı 2: Gaussian Gürültü Saldırısı
════════════════════════════════════════════════════════════

Fikir: Feature değerlerine küçük gürültü ekle
Gerçek hayat: Saldırgan modelin feature'larını tam bilmiyor, rastgele deniyor

  Adversarial Değerlendirme: Random Forest
  Normal Accuracy    : 0.9271
  Adversarial Accuracy: 0.9271
  Accuracy Düşüşü    : 0.0000 (0.0%)
  F1 Düşüşü          : 0.0000
  Saldırı Başarısı   : 0.00%
  Robustness Skoru   : 1.0000

  Adversarial Değerlendirme: XGBoost
  Normal Accuracy    : 0.9392
  Adversarial Accuracy: 0.9470
  Accuracy Düşüşü    : -0.0078 (-0.8%)
  F1 Düşüşü          : -0.0082
  Saldırı Başarısı   : -1.66%
  Robustness Skoru   : 1.0078

  Adversarial Değerlendirme: Logistic Regression
  Normal Accuracy    : 0.8900
  Adversarial Accuracy: 0.8895
  Accuracy Düşüşü    : 0.0005 (0.1%)
  F1 Düşüşü          : 0.0006
  Saldırı Başarısı   : 0.12%
  Robustness Skoru   : 0.9995

              

## 🔴 Saldırı 3: Kombine (En Güçlü)

In [6]:
print_section('Saldırı 3: Kombine Saldırı (Minimal + Gaussian)')
print('Fikir: İki saldırıyı birleştir - daha güçlü etki')

X_adv_combined = attacker.combined_attack(X_test, feature_names, y_test)

results_combined = []
for name, model in baseline_models.items():
    r = evaluator.evaluate(model, X_test, X_adv_combined, y_test, name)
    results_combined.append(r)

df_combined = pd.DataFrame(results_combined).set_index('model')
print('\n', df_combined[['acc_normal', 'acc_adversarial', 'acc_drop', 'attack_success_rate']].to_string())


════════════════════════════════════════════════════════════
  Saldırı 3: Kombine Saldırı (Minimal + Gaussian)
════════════════════════════════════════════════════════════

Fikir: İki saldırıyı birleştir - daha güçlü etki
   [1/2] Minimal feature manipülasyonu uygulanıyor...
   [2/2] Gaussian gürültü ekleniyor...

  Adversarial Değerlendirme: Random Forest
  Normal Accuracy    : 0.9271
  Adversarial Accuracy: 0.9038
  Accuracy Düşüşü    : 0.0233 (2.5%)
  F1 Düşüşü          : 0.0257
  Saldırı Başarısı   : 5.05%
  Robustness Skoru   : 0.9767

  Adversarial Değerlendirme: XGBoost
  Normal Accuracy    : 0.9392
  Adversarial Accuracy: 0.9409
  Accuracy Düşüşü    : -0.0017 (-0.2%)
  F1 Düşüşü          : -0.0018
  Saldırı Başarısı   : -0.36%
  Robustness Skoru   : 1.0017

  Adversarial Değerlendirme: Logistic Regression
  Normal Accuracy    : 0.8900
  Adversarial Accuracy: 0.7830
  Accuracy Düşüşü    : 0.1071 (12.0%)
  F1 Düşüşü          : 0.1387
  Saldırı Başarısı   : 25.00%
  Robustness Sk

## 📊 Saldırı Karşılaştırması

In [7]:
# Tüm saldırıların özet karşılaştırması
fig, axes = plt.subplots(1, 3, figsize=(18, 6))

attack_results = [
    ('Minimal Manipulation', results_minimal),
    ('Gaussian Noise', results_gaussian),
    ('Combined Attack', results_combined),
]

colors = ['#2196F3', '#FF9800', '#4CAF50']

for ax, (attack_name, results) in zip(axes, attack_results):
    models = [r['model'] for r in results]
    acc_normal = [r['acc_normal'] for r in results]
    acc_adv    = [r['acc_adversarial'] for r in results]
    
    x = np.arange(len(models))
    width = 0.35
    
    ax.bar(x - width/2, acc_normal, width, label='Normal', color='#4CAF50', alpha=0.8)
    ax.bar(x + width/2, acc_adv, width, label='Adversarial', color='#F44336', alpha=0.8)
    ax.set_title(attack_name, fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels([m.split()[0] for m in models], rotation=10)
    ax.set_ylim(0, 1.1)
    ax.set_ylabel('Accuracy')
    ax.legend()
    
    # Düşüş oklarını ekle
    for xi, (n, a) in enumerate(zip(acc_normal, acc_adv)):
        drop = n - a
        ax.annotate(f'↓{drop:.3f}', xy=(xi + width/2, a), 
                    xytext=(xi + width/2, a - 0.05),
                    fontsize=8, ha='center', color='red')

plt.suptitle('Adversarial Saldırı Etkisi: Normal vs Adversarial Accuracy', 
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'attack_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [8]:
# Robustness karşılaştırması
robustness_data = {}
for attack_name, results in attack_results:
    for r in results:
        if r['model'] not in robustness_data:
            robustness_data[r['model']] = {}
        robustness_data[r['model']][attack_name] = r['robustness_score']

rob_df = pd.DataFrame(robustness_data).T
print('\n🛡️ Robustness Skorları (1.0 = tam dayanıklı, 0 = savunmasız):')
print(rob_df.to_string())

# Radar chart yerine heatmap
fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(rob_df, annot=True, fmt='.3f', cmap='RdYlGn',
            vmin=0.5, vmax=1.0, ax=ax)
ax.set_title('Model Robustness Matrisi\n(Yeşil = Dayanıklı, Kırmızı = Savunmasız)', fontsize=13)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'robustness_matrix.png', dpi=150, bbox_inches='tight')
plt.show()


🛡️ Robustness Skorları (1.0 = tam dayanıklı, 0 = savunmasız):
                     Minimal Manipulation  Gaussian Noise  Combined Attack
Random Forest                      0.9766          1.0000           0.9767
XGBoost                            0.9843          1.0078           1.0017
Logistic Regression                0.8983          0.9995           0.8929


## 🔬 Feature Sensitivity Analizi

In [9]:
print_section('Feature Sensitivity Analizi')
print('Hangi feature manipülasyonu modeli en çok etkiliyor?')

# En iyi modelde analiz yap (RF)
best_model = baseline_models['Random Forest']
sensitivity_df = evaluator.feature_sensitivity_analysis(
    best_model, X_test, feature_names, y_test, n_samples=200
)

print('\nTop 10 En Etkili Feature Manipülasyonu:')
print(sensitivity_df.head(10).to_string())


════════════════════════════════════════════════════════════
  Feature Sensitivity Analizi
════════════════════════════════════════════════════════════

Hangi feature manipülasyonu modeli en çok etkiliyor?

🔬 Feature sensitivity analizi başlıyor...
✅ Sensitivity analizi tamamlandı. En etkili feature: is_https

Top 10 En Etkili Feature Manipülasyonu:
                feature  baseline_acc  modified_acc  acc_drop impact
7              is_https          0.93         0.790     0.140   HIGH
12        domain_length          0.93         0.910     0.020    LOW
0            url_length          0.93         0.920     0.010    LOW
4           url_entropy          0.93         0.925     0.005    LOW
5        domain_entropy          0.93         0.925     0.005    LOW
3           digit_ratio          0.93         0.925     0.005    LOW
2                num_at          0.93         0.930     0.000    LOW
1           num_hyphens          0.93         0.930     0.000    LOW
9         has_homoglyph   

In [10]:
# Sensitivity görselleştir
top_sens = sensitivity_df.head(15)
colors_map = {'HIGH': '#F44336', 'MEDIUM': '#FF9800', 'LOW': '#4CAF50'}
bar_colors = [colors_map[i] for i in top_sens['impact']]

fig, ax = plt.subplots(figsize=(12, 7))
ax.barh(top_sens['feature'][::-1], top_sens['acc_drop'][::-1], 
        color=bar_colors[::-1], alpha=0.85)
ax.set_title('Feature Sensitivity: Hangi Feature Saldırısı En Çok Etkiliyor?\n(Random Forest)', 
             fontsize=12)
ax.set_xlabel('Accuracy Düşüşü')
ax.axvline(x=0.1, color='orange', linestyle='--', label='Orta etki (0.1)')
ax.axvline(x=0.2, color='red', linestyle='--', label='Yüksek etki (0.2)')
ax.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'feature_sensitivity.png', dpi=150, bbox_inches='tight')
plt.show()

In [11]:
# Sonuçları kaydet
import pickle

attack_results_to_save = {
    'minimal': results_minimal,
    'gaussian': results_gaussian, 
    'combined': results_combined,
    'sensitivity': sensitivity_df.to_dict(),
}

np.save('../data/processed/X_adv_minimal.npy', X_adv_minimal)
np.save('../data/processed/X_adv_gaussian.npy', X_adv_gaussian)
np.save('../data/processed/X_adv_combined.npy', X_adv_combined)

with open('../data/processed/attack_results.json', 'w') as f:
    import json
    # DataFrame'i dict'e çevir
    serializable = {k: (v if not isinstance(v, dict) else v) 
                    for k, v in attack_results_to_save.items()}
    json.dump(serializable, f, default=str, indent=2)

print('✅ Saldırı sonuçları kaydedildi')
print('\n✅ Notebook 4 tamamlandı!')
print('   Sonraki adım: 5_defense_mechanism.ipynb')

✅ Saldırı sonuçları kaydedildi

✅ Notebook 4 tamamlandı!
   Sonraki adım: 5_defense_mechanism.ipynb
